# 🏡 AI Room Makeover — GPU Inference Server (Google Colab)

Run high-performance Image-to-Image Generative AI models (`SDXL-Turbo` or `SD-Turbo`) on a free Google Colab GPU to power your local **AI Room Makeover** app!

### ⚡ Quick Start:
1. Go to menu: **Runtime → Change runtime type → T4 GPU → Save**.
2. Run all cells (**Runtime → Run all** or press `Ctrl + F9`).
3. Copy the generated `COLAB_INFERENCE_URL` from the bottom cell and paste it into your local `backend/.env` file.

In [ ]:
#@title 1. Install Dependencies & Tunnels { display-mode: "form" }
import sys
import subprocess

print("📦 Installing required libraries (Diffusers, Transformers, FastAPI, Accelerate)...")
!pip install -q diffusers transformers accelerate safetensors fastapi uvicorn pyngrok pydantic pillow

# Download Cloudflare tunnel binary (100% free, zero login required)
print("🌐 Setting up Cloudflare tunnel tool...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Setup complete!")

In [ ]:
#@title 2. Check GPU & Load Diffusion Model { display-mode: "form" }
import torch
from diffusers import AutoPipelineForImage2Image

if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU detected! Please go to Runtime > Change runtime type > select 'T4 GPU' and re-run.")

gpu_name = torch.cuda.get_device_name(0)
gpu_vram = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
print(f"✅ Active GPU: {gpu_name} ({gpu_vram} GiB VRAM)")

# Model selection form
MODEL_CHOICE = "stabilityai/sdxl-turbo" #@param ["stabilityai/sdxl-turbo", "stabilityai/sd-turbo"] {allow-input: true}

print(f"\n⏳ Loading '{MODEL_CHOICE}' into GPU VRAM (takes ~20-30 seconds)...")

pipe = AutoPipelineForImage2Image.from_pretrained(
    MODEL_CHOICE,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")

pipe.enable_attention_slicing()
print(f"🎉 Model '{MODEL_CHOICE}' successfully loaded on CUDA!")

In [ ]:
#@title 3. Start Inference Server & Public Tunnel { display-mode: "form" }
import base64
import datetime
import io
import json
import os
import re
import secrets
import subprocess
import threading
import time
import traceback
import urllib.request
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from PIL import Image

# Configuration parameters via Colab form
AUTH_TOKEN = "EtUrHjUhuuziUbx6bzN7h3_2O8GGKNdJQmKw-c7G2iU" #@param {type:"string"}
TUNNEL_PROVIDER = "Cloudflare (Zero Setup - Recommended)" #@param ["Cloudflare (Zero Setup - Recommended)", "Ngrok (Fastest, requires authtoken)", "LocalTunnel"]
NGROK_AUTHTOKEN = "" #@param {type:"string"}

class GenerationRequest(BaseModel):
    image_base64: str = Field(min_length=1, max_length=20_000_000)
    prompt: str = Field(min_length=1, max_length=5_000)
    strength: float = Field(default=0.65, ge=0.05, le=1.0)
    steps: int = Field(default=4, ge=1, le=10)

app = FastAPI(title="AI Room Makeover Inference Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
generation_lock = threading.Lock()

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0), "model": MODEL_CHOICE}

@app.post("/api/try-on")
def generate(request: GenerationRequest, authorization: str | None = Header(default=None)):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print("\n" + "━" * 60)
    print(f"📥 [{timestamp}] INCOMING REQUEST RECEIVED ON COLAB GPU!")
    print(f"   📝 Prompt: '{request.prompt}'")
    print(f"   ⚙️  Strength: {request.strength} | Steps: {request.steps}")
    print(f"   📦 Payload Base64 Size: {len(request.image_base64) / 1024:.1f} KB")

    if AUTH_TOKEN.strip():
        expected_auth = f"Bearer {AUTH_TOKEN.strip()}"
        if authorization is None or not secrets.compare_digest(authorization.strip(), expected_auth):
            print(f"   ❌ AUTHENTICATION ERROR: Token mismatch or missing header!")
            print(f"      Provided: '{authorization}'")
            print(f"      Expected: '{expected_auth}'")
            print("━" * 60 + "\n")
            raise HTTPException(status_code=401, detail="Invalid inference token")
        print("   🔑 Auth token verified successfully.")

    t_start = time.time()
    try:
        raw_b64 = request.image_base64
        if "," in raw_b64:
            raw_b64 = raw_b64.split(",", 1)[1]
        image_bytes = base64.b64decode(raw_b64, validate=True)
        with Image.open(io.BytesIO(image_bytes)) as source:
            image = source.convert("RGB")
        print(f"   🖼️  Input Image Decoded: {image.size[0]}x{image.size[1]} ({len(image_bytes)/1024:.1f} KB)")
    except Exception as exc:
        print(f"   ❌ Image decode error: {exc}")
        print("━" * 60 + "\n")
        raise HTTPException(status_code=400, detail=f"Could not read image: {exc}") from exc

    # Optimal sizing (multiples of 8)
    max_dim = 1024 if "sdxl" in MODEL_CHOICE.lower() else 512
    image.thumbnail((max_dim, max_dim), Image.Resampling.LANCZOS)
    w = max(8, (image.width // 8) * 8)
    h = max(8, (image.height // 8) * 8)
    image = image.resize((w, h), Image.Resampling.LANCZOS)

    is_turbo = "turbo" in MODEL_CHOICE.lower()
    guidance = 0.0 if is_turbo else 7.5
    inference_steps = min(request.steps, 4) if is_turbo else max(request.steps, 15)

    print(f"   🎨 Running Diffusion Model on GPU ({w}x{h}, steps={inference_steps}, guidance={guidance})...")
    gen_start = time.time()
    try:
        with generation_lock:
            result = pipe(
                prompt=request.prompt,
                image=image,
                strength=request.strength,
                num_inference_steps=inference_steps,
                guidance_scale=guidance,
            ).images[0]

        gen_time = time.time() - gen_start
        print(f"   ✨ AI Image Generated in {gen_time:.2f} seconds! Output size: {result.size[0]}x{result.size[1]}")

        output = io.BytesIO()
        result.save(output, format="PNG")
        output_bytes = output.getvalue()
        encoded_result = base64.b64encode(output_bytes).decode("ascii")
        torch.cuda.empty_cache()

        total_time = time.time() - t_start
        print(f"   📤 Returning generated image: {len(output_bytes)/1024:.1f} KB (Total processing time: {total_time:.2f}s)")
        print("━" * 60 + "\n")
        return {"image_base64": encoded_result}
    except Exception as exc:
        torch.cuda.empty_cache()
        print(f"   ❌ Generation failed with exception: {exc}")
        traceback.print_exc()
        print("━" * 60 + "\n")
        raise HTTPException(status_code=500, detail=str(exc)) from exc

# Run FastAPI background server on port 7860 with info logging
def run_uvicorn():
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=7860, log_level="info")

threading.Thread(target=run_uvicorn, daemon=True).start()

# Wait for local server to be responsive
server_ready = False
for _ in range(25):
    try:
        with urllib.request.urlopen("http://127.0.0.1:7860/health", timeout=1) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(0.5)

if not server_ready:
    raise RuntimeError("FastAPI server failed to start on port 7860.")

# Start Public Tunnel
public_url = None
print("\n🌐 Establishing public tunnel...")

if "Cloudflare" in TUNNEL_PROVIDER:
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for _ in range(40):
        line = cf_proc.stdout.readline()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
        time.sleep(0.5)

elif "Ngrok" in TUNNEL_PROVIDER and NGROK_AUTHTOKEN.strip():
    from pyngrok import ngrok
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    tunnel = ngrok.connect(7860)
    public_url = tunnel.public_url

else:
    lt_proc = subprocess.Popen(
        ["npx", "-y", "localtunnel", "--port", "7860"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for _ in range(30):
        line = lt_proc.stdout.readline()
        if "url is:" in line.lower():
            public_url = line.strip().split()[-1]
            if not public_url.startswith("http"):
                public_url = f"https://{public_url}"
            break
        time.sleep(1)

if not public_url:
    raise RuntimeError("Failed to establish tunnel. Please re-run the cell or try another tunnel provider.")

endpoint_url = f"{public_url}/api/try-on"

print("\n" + "=" * 65)
print("🎉 GOOGLE COLAB GPU SERVER IS ONLINE AND READY!")
print("=" * 65)
print("\n📋 Copy and paste these lines into your local 'backend/.env' file:\n")
print(f"USE_LOCAL_MODEL=auto")
print(f"COLAB_INFERENCE_URL={endpoint_url}")
if AUTH_TOKEN.strip():
    print(f"COLAB_INFERENCE_TOKEN={AUTH_TOKEN.strip()}")
print("\n" + "=" * 65)
print("⚠️ KEEP THIS TAB RUNNING in your browser while using the app.")
print("=" * 65)